# DD Startup Analysis - Run Interface

This notebook provides a simple interface to run DD startup simulations without using the command line.

**Equivalent to running:**
```bash
python -m ddstartup <param_file> <config_file>
```

## Quick Start
1. Choose your parameter file and configuration in the cell below
2. Run all cells to execute the analysis
3. Results will be saved to the `outputs/` directory

## Configuration

**Select your files here:**

In [ ]:
# ============================================================================
# CHOOSE YOUR FILES HERE
# ============================================================================

# Parameter file (in inputs/ directory)
# Options: 'params', 'params_test', 'params_noPaux', '6params', etc.
PARAM_FILE = 'params_noPaux'  # Can use .yaml or .py extension (optional)

# Configuration file (in inputs/ directory)
# Options: 'parametric_tseeded', 'parametric_lump', 'sobol_tseeded', 'sobol_lump'
CONFIG_FILE = 'parametric_tseeded'  # Can use .yaml extension (optional)

# Additional options
VERBOSE = True      # Show detailed progress
DRY_RUN = False     # Only validate, don't run (set to True for testing)

print(f"Selected configuration:")
print(f"  Parameter file: {PARAM_FILE}")
print(f"  Config file: {CONFIG_FILE}")
print(f"  Verbose: {VERBOSE}")
print(f"  Dry run: {DRY_RUN}")

## Run Analysis

This cell runs `ddstartup.main()` with your selected configuration.

In [ ]:
import sys
import warnings

# Suppress scipy integration warnings
warnings.filterwarnings("ignore", module="scipy.integrate")

# Import ddstartup
from ddstartup.main import main

def run_ddstartup(param_file, config_file, verbose=True, dry_run=False):
    """
    Run ddstartup.main with specified parameters.
    
    This is equivalent to running:
    python -m ddstartup <param_file> <config_file> [--verbose] [--dry-run]
    """
    # Save original argv
    original_argv = sys.argv.copy()
    
    try:
        # Set up arguments
        sys.argv = ['ddstartup', param_file, config_file]
        if verbose:
            sys.argv.append('--verbose')
        if dry_run:
            sys.argv.append('--dry-run')
        
        print(f"\nRunning: ddstartup {param_file} {config_file}" + 
              (" --verbose" if verbose else "") + 
              (" --dry-run" if dry_run else ""))
        print("=" * 80)
        
        # Run main
        result = main()
        
        print("=" * 80)
        if result == 0:
            print("✅ Analysis completed successfully!")
        else:
            print("❌ Analysis failed with errors.")
        
        return result
        
    finally:
        # Restore original argv
        sys.argv = original_argv

# Run the analysis
exit_code = run_ddstartup(
    param_file=PARAM_FILE,
    config_file=CONFIG_FILE,
    verbose=VERBOSE,
    dry_run=DRY_RUN
)

## Postprocessing (Optional)

Generate plots and analyze results using a postprocessing configuration file.

In [ ]:
# ============================================================================
# POSTPROCESSING CONFIGURATION
# ============================================================================

# Postprocess configuration file (in inputs/ directory)
# Options: 'postprocess_config', 'postprocess_quick'
POSTPROCESS_CONFIG = 'postprocess_config'  # Can use .yaml extension (optional)

# Set to True to run postprocessing automatically after analysis
RUN_POSTPROCESSING = True

print(f"Postprocessing configuration:")
print(f"  Config file: {POSTPROCESS_CONFIG}")
print(f"  Auto-run: {RUN_POSTPROCESSING}")

# ============================================================================
# RUN POSTPROCESSING
# ============================================================================

if RUN_POSTPROCESSING and exit_code == 0:
    print("\n" + "=" * 80)
    print("STARTING POSTPROCESSING")
    print("=" * 80)
    
    # Import postprocessing CLI
    from ddstartup.postprocessing.cli import main as postprocess_main
    
    # Save original argv
    original_argv = sys.argv.copy()
    
    try:
        # Set up arguments for postprocessing
        sys.argv = ['ddstartup.postprocessing', POSTPROCESS_CONFIG]
        
        print(f"\nRunning: ddstartup.postprocessing {POSTPROCESS_CONFIG}")
        print("=" * 80)
        
        # Run postprocessing
        postprocess_result = postprocess_main()
        
        print("=" * 80)
        if postprocess_result == 0:
            print("✅ Postprocessing completed successfully!")
        else:
            print("❌ Postprocessing failed with errors.")
            
    finally:
        # Restore original argv
        sys.argv = original_argv
        
elif RUN_POSTPROCESSING and exit_code != 0:
    print("\n⚠️ Skipping postprocessing because analysis failed.")
else:
    print("\nℹ️ Postprocessing disabled. Set RUN_POSTPROCESSING = True to enable.")